# Predictor Viral: ¿Este video va a reventar?

Pipeline completo de ML aplicando todo lo aprendido en los 4 notebooks anteriores.
Un modelo que predice si un video será viral en TikTok **o** Instagram.

**Dataset:** Datos propios @aroaxinping (15 TikToks + 68 Reels)  
**Autor:** @aroaxinping  
**Fecha:** Abril 2026

> **Nota:** Con ~83 posts el dataset es pequeño. Este notebook es un ejercicio real
> de ML con datos propios — no un modelo de producción. La honestidad sobre las
> limitaciones es parte del análisis.

---
## 0. Configuración del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, accuracy_score,
)
import xgboost as xgb

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'axes.titlecolor':  '#ffffff',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

ACCENT  = '#e85d04'
ACCENT2 = '#6a9ad4'
ACCENT3 = '#2dc653'
WARN    = '#f4d03f'

print('Entorno listo.')

---
## 1. Datos

| Plataforma | Fuente | Posts |
|---|---|---|
| TikTok | Export manual TikTok Studio | ~15 videos |
| Instagram | Export Meta Business Suite | ~68 reels |

> **Nota:** El script `src/prepare_social_data.py` unifica ambos datasets en un formato común.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

DATA_PATH = Path('../data/processed/social_unified.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH, parse_dates=['date'])
    es_sintetico = False
    print(f'[OK] Datos cargados: {len(df)} posts')
else:
    print('[INFO] Datos no encontrados. Generando dataset sintético...')
    print('[TIP]  Ejecuta: python src/prepare_social_data.py')
    from prepare_social_data import generate_synthetic_social, define_viral
    df = generate_synthetic_social()
    df = define_viral(df)
    es_sintetico = True

if es_sintetico:
    print('\n⚠️  AVISO: Datos sintéticos.')

print(f'\n--- Resumen ---')
print(df.groupby('platform').agg(
    posts=('views', 'count'),
    views_median=('views', 'median'),
    views_mean=('views', 'mean'),
    virales=('is_viral', 'sum'),
).to_markdown())
df.head()

---
## 2. Exploración

> **Pregunta:** ¿Qué hace que un video sea viral vs normal en cada plataforma?

In [ ]:
# 2.1 Distribución de views por plataforma
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, platform in zip(axes, ['tiktok', 'instagram']):
    data = df[df['platform'] == platform]
    for val, color, label in [(0, ACCENT2, 'Normal'), (1, ACCENT, 'Viral')]:
        subset = data[data['is_viral'] == val]['views']
        if len(subset) > 0:
            ax.hist(subset, bins=15, alpha=0.6, color=color, label=f'{label} ({len(subset)})')
    median = data['views'].median()
    threshold = median * 3
    ax.axvline(threshold, ls='--', color=WARN, lw=1.5, label=f'Threshold = {threshold:,.0f}')
    ax.set(xlabel='Views', ylabel='Count', title=f'{platform.title()} — Distribución de Views')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# 2.2 Engagement rate: viral vs normal
fig, ax = plt.subplots(figsize=(10, 5))

for platform, offset in [('tiktok', -0.15), ('instagram', 0.15)]:
    data = df[df['platform'] == platform]
    for val, color in [(0, ACCENT2), (1, ACCENT)]:
        subset = data[data['is_viral'] == val]['engagement_rate']
        label = f'{platform} - {"Viral" if val else "Normal"}'
        ax.scatter(
            [val + offset] * len(subset), subset,
            alpha=0.6, s=40, color=color, label=label,
        )

ax.set(xlabel='Is Viral', ylabel='Engagement Rate (%)', title='Engagement Rate por Viralidad')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Normal', 'Viral'])
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 Topic vs viralidad
topic_col = 'topic_clean' if 'topic_clean' in df.columns else 'topic'
topic_viral = df.groupby(topic_col)['is_viral'].agg(['mean', 'count']).sort_values('mean', ascending=True)
topic_viral.columns = ['viral_rate', 'n_posts']
topic_viral = topic_viral[topic_viral['n_posts'] >= 2]  # min 2 posts

fig, ax = plt.subplots(figsize=(10, 5))
colors = [ACCENT if r > 0.3 else ACCENT2 for r in topic_viral['viral_rate']]
bars = ax.barh(topic_viral.index, topic_viral['viral_rate'] * 100, color=colors, edgecolor='#333')
for bar, n in zip(bars, topic_viral['n_posts']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'n={n}', va='center', color='#999', fontsize=9)
ax.set(xlabel='Viral Rate (%)', title='% Viralidad por Topic')
plt.tight_layout()
plt.show()

---
## 3. Feature Engineering + Pipeline

> **Pregunta:** ¿Qué features podemos usar que estén disponibles ANTES de publicar?

In [ ]:
# 3.1 Definir features (solo las que sabes antes de publicar)
# NO usar: views, likes, comments, shares, saves, engagement_rate (son outcomes)

NUM_FEATURES = ['duration_sec', 'title_length', 'num_hashtags', 'num_emojis']
CAT_FEATURES = ['platform', topic_col]

# Añadir day_of_week si existe
if 'day_of_week' in df.columns:
    NUM_FEATURES.append('day_of_week')
if 'is_weekend' in df.columns:
    NUM_FEATURES.append('is_weekend')

ALL_FEATURES = NUM_FEATURES + CAT_FEATURES
TARGET = 'is_viral'

# Verificar que existen
available_num = [f for f in NUM_FEATURES if f in df.columns]
available_cat = [f for f in CAT_FEATURES if f in df.columns]
print(f'Features numéricas: {available_num}')
print(f'Features categóricas: {available_cat}')

# Limpiar NaN
df_model = df[available_num + available_cat + [TARGET]].dropna()
print(f'\nMuestras válidas: {len(df_model)} / {len(df)}')
print(f'Virales: {df_model[TARGET].sum()} ({df_model[TARGET].mean()*100:.1f}%)')

In [ ]:
# 3.2 Pipeline con ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), available_num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), available_cat),
    ]
)

X = df_model[available_num + available_cat]
y = df_model[TARGET]

print(f'X shape: {X.shape}')
print(f'y balance: {y.value_counts().to_dict()}')

---
## 4. Modelo: Comparación de 3 algoritmos

> **Pregunta:** ¿Qué modelo generaliza mejor con tan pocos datos? ¿Logistic, RF o XGBoost?

In [ ]:
# 4.1 Stratified K-Fold (k=5)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'Random Forest': Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)),
    ]),
    'XGBoost': Pipeline([
        ('prep', preprocessor),
        ('clf', xgb.XGBClassifier(
            n_estimators=50, max_depth=3, learning_rate=0.1,
            random_state=42, verbosity=0, use_label_encoder=False,
            eval_metric='logloss',
        )),
    ]),
}

results = {}
print('--- Cross-Validation (5-fold stratified) ---\n')
for name, pipe in models.items():
    cv_acc = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
    cv_f1 = cross_val_score(pipe, X, y, cv=cv, scoring='f1')
    results[name] = {'accuracy': cv_acc, 'f1': cv_f1}
    print(f'{name:25s}  Acc={cv_acc.mean():.3f}±{cv_acc.std():.3f}  F1={cv_f1.mean():.3f}±{cv_f1.std():.3f}')

In [ ]:
# 4.2 Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ['accuracy', 'f1']):
    data_plot = [results[m][metric] for m in models]
    bp = ax.boxplot(data_plot, labels=list(models.keys()), patch_artist=True, widths=0.5)
    for patch, color in zip(bp['boxes'], [ACCENT, ACCENT2, ACCENT3]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    for element in ['whiskers', 'caps', 'medians']:
        for line in bp[element]:
            line.set_color('#e0e0e0')
    ax.set(ylabel=metric.upper(), title=f'{metric.upper()} — 5-Fold CV')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

---
## 5. Mejor modelo: Feature Importance

> **Pregunta:** ¿Qué factores hacen que MI contenido sea viral?

In [ ]:
# 5.1 Entrenar el mejor modelo en todo el dataset
# Elegir el de mejor F1 medio
best_name = max(results, key=lambda m: results[m]['f1'].mean())
best_pipe = models[best_name]
best_pipe.fit(X, y)

print(f'Mejor modelo: {best_name}')
print(f'F1 CV medio: {results[best_name]["f1"].mean():.3f}')

In [ ]:
# 5.2 Feature importance
# Obtener nombres de features después del preprocessing
try:
    cat_names = best_pipe.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(available_cat).tolist()
except:
    cat_names = [f'cat_{i}' for i in range(10)]

all_feature_names = available_num + cat_names

clf = best_pipe.named_steps['clf']
if hasattr(clf, 'feature_importances_'):
    importances = clf.feature_importances_
elif hasattr(clf, 'coef_'):
    importances = np.abs(clf.coef_[0])
else:
    importances = np.zeros(len(all_feature_names))

# Ajustar longitud si no coincide
if len(importances) != len(all_feature_names):
    all_feature_names = [f'feature_{i}' for i in range(len(importances))]

fi = pd.Series(importances, index=all_feature_names).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi.index, fi.values, color=ACCENT, edgecolor='#333')
ax.set(xlabel='Importance', title=f'Feature Importance — {best_name}')
plt.tight_layout()
plt.show()

print(f'\nTop 3 features: {fi.tail(3).index.tolist()[::-1]}')

---
## 6. SHAP (si disponible)

> **Pregunta:** ¿Cómo contribuye cada feature a cada predicción individual?

In [ ]:
# 6.1 SHAP values
try:
    import shap

    # Transformar X
    X_transformed = best_pipe.named_steps['prep'].transform(X)
    if hasattr(X_transformed, 'toarray'):
        X_transformed = X_transformed.toarray()
    X_transformed = pd.DataFrame(X_transformed, columns=all_feature_names)

    explainer = shap.TreeExplainer(clf) if hasattr(clf, 'feature_importances_') else shap.LinearExplainer(clf, X_transformed)
    shap_values = explainer.shap_values(X_transformed)

    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # clase positiva

    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X_transformed, show=False)
    plt.title('SHAP — ¿Qué hace viral a TU contenido?', color='#fff', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f'SHAP no disponible: {e}')
    print('pip install shap para habilitar esta sección.')

---
## 7. Análisis por plataforma

> **Pregunta:** ¿El mismo contenido funciona igual en TikTok que en Instagram?

In [ ]:
# 7.1 Métricas clave por plataforma
platform_stats = df.groupby('platform').agg(
    posts=('views', 'count'),
    views_median=('views', 'median'),
    views_mean=('views', 'mean'),
    engagement_mean=('engagement_rate', 'mean'),
    viral_rate=('is_viral', 'mean'),
).round(2)
platform_stats['viral_rate'] = (platform_stats['viral_rate'] * 100).round(1)

print('--- Comparativa por Plataforma ---')
print(platform_stats.to_markdown())

# Topic performance cruzado
print('\n--- Viral Rate por Topic y Plataforma ---')
cross = df.groupby([topic_col, 'platform'])['is_viral'].agg(['mean', 'count'])
cross.columns = ['viral_rate', 'n']
cross['viral_rate'] = (cross['viral_rate'] * 100).round(1)
cross = cross[cross['n'] >= 2]
print(cross.to_markdown())

---
## 8. Función de predicción

> Una función que puedes usar antes de publicar para estimar probabilidad de viralidad.

In [ ]:
# 8.1 Función predict_viral
def predict_viral(
    platform: str,
    duration_sec: int,
    topic: str,
    title: str = "",
    day_of_week: int = 2,  # miércoles por defecto
):
    """
    Predice probabilidad de viralidad para un video.
    
    Args:
        platform: 'tiktok' o 'instagram'
        duration_sec: duración en segundos
        topic: tema del contenido
        title: título/descripción del video
        day_of_week: 0=lunes, 6=domingo
    """
    row = {
        'duration_sec': duration_sec,
        'title_length': len(title),
        'num_hashtags': title.count('#'),
        'num_emojis': sum(1 for c in title if ord(c) > 0x1F600),
        'platform': platform,
        topic_col: topic,
    }
    if 'day_of_week' in available_num:
        row['day_of_week'] = day_of_week
    if 'is_weekend' in available_num:
        row['is_weekend'] = 1 if day_of_week >= 5 else 0
    
    input_df = pd.DataFrame([row])
    prob = best_pipe.predict_proba(input_df[available_num + available_cat])[0, 1]
    pred = 'VIRAL' if prob >= 0.5 else 'Normal'
    
    print(f'  Plataforma: {platform}')
    print(f'  Duración:   {duration_sec}s')
    print(f'  Topic:      {topic}')
    print(f'  Predicción: {pred} (probabilidad = {prob:.1%})')
    return prob

print('--- Ejemplos de predicción ---\n')

print('Video 1: TikTok tech humor corto')
predict_viral('tiktok', 15, 'tech_humor', 'me pagan por romper cosas #programacion #techhumor')

print('\nVideo 2: Instagram setup largo')
predict_viral('instagram', 45, 'setup', 'nuevo setup de escritorio #setup #desksetup')

print('\nVideo 3: TikTok data')
predict_viral('tiktok', 25, 'data', 'tu playlist dice más de ti que tu CV #datascience #spotify')

---
## 9. Limitaciones

Ser honesta sobre qué puede y qué NO puede hacer este modelo:

| Limitación | Impacto | Mitigación |
|---|---|---|
| **~83 posts** | Muestra muy pequeña para ML robusto | Stratified K-Fold, modelos simples, regularización |
| **Solo features pre-publicación** | No captura calidad del video, edición, audio | Es lo que podemos controlar — el modelo es útil para planificar |
| **Periodo corto** (feb-abr 2026) | No captura estacionalidad ni cambios de algoritmo | Actualizar con más datos cada mes |
| **Definición de "viral" arbitraria** | Threshold de 3x mediana es discutible | Probar con 2x, 5x, percentil 75 |
| **Plataformas distintas** | TikTok y IG tienen algoritmos muy diferentes | La feature `platform` captura parte de esto |

---
## 10. Síntesis y conclusiones

In [ ]:
# 10.1 Tabla resumen de modelos
resumen = pd.DataFrame([
    {
        'Modelo': name,
        'Accuracy (CV)': f"{results[name]['accuracy'].mean():.3f} ± {results[name]['accuracy'].std():.3f}",
        'F1 (CV)': f"{results[name]['f1'].mean():.3f} ± {results[name]['f1'].std():.3f}",
    }
    for name in models
])
print(resumen.to_markdown(index=False))

print(f'\n--- Conclusión ---')
print(f'Mejor modelo: {best_name} (F1 = {results[best_name]["f1"].mean():.3f})')
print(f'\nCon ~83 posts, cualquier resultado es orientativo, no definitivo.')
print(f'Lo que SÍ se puede extraer:')
print(f'  - Qué topics tienen más potencial viral en cada plataforma')
print(f'  - Si la duración del video importa')
print(f'  - Una estimación antes de publicar (mejor que intuición pura)')
print(f'\nPróximo paso: seguir recopilando datos y re-entrenar cada mes.')
print(f'Con 200+ posts el modelo será significativamente más fiable.')